# API Discovery RAG — Ingestion, Retrieval & Examination (Phase 1)
**Use case:** an internal *API discovery* assistant. A developer asks *'which API refunds a payment?'* in plain language and the system returns the right endpoint. This notebook builds that end to end on Chroma, then examines the collection the way you would in **Chroma Studio**.

**Phase 1 (this notebook):** discover APIs by *purpose* + *endpoint*, filter by `domain` and `method`. **Phase 2 (later, separate):** request/response schemas.

Designed as a research artifact: every step states *why* it's built that way, so it doubles as a template you can defend in a design review.

## Why this dataset is shaped the way it is (design rationale)
- **We embed the API's PURPOSE, not its path.** Developers search by intent ('cancel a card'), not by URL. Embedding a natural-language purpose statement is what makes semantic discovery work — embedding `/v1/cards/{id}/freeze` would not.
- **`domain` and `method` are metadata, not text.** They're exact-match facets you filter on (show only `payments`, only `POST`), so they belong in metadata where Chroma can filter — not buried in the embedded text where they'd just add noise.
- **Planted structure for examination:** a v1/v2 near-duplicate, a deprecated endpoint, a cross-domain overlap, and a semantic collision — so the Studio examination section has real findings to surface, like a real catalog would.

## Setup
`USE_MOCK=True` runs offline to prove the plumbing. Set `False` for real in-house Jina embeddings (discovery quality is much better with the real model).

In [ ]:
USE_MOCK = True   # set False for real InHouseEmbeddings()

import chromadb, pandas as pd, numpy as np, re

if USE_MOCK:
    _STOP = set("the a an to of and or is are be for in on at by with from your you we "
                "our it its as within once after before per must can may that this".split())
    class MockEmbedder:
        def _vec(self, t):
            v = np.zeros(256)
            for w in re.findall(r"[a-z0-9]+", t.lower()):
                if w not in _STOP and len(w) > 2:
                    v[abs(hash(w)) % 256] += 1
            n = np.linalg.norm(v); return (v/n if n else v)
        def __call__(self, input):
            if isinstance(input, str): input = [input]
            return [self._vec(t).tolist() for t in input]
        def embed_documents(self, texts): return self(texts)
        def embed_query(self, text): return self._vec(text).tolist()
    embedder = MockEmbedder()
    print("MOCK embedder (word-overlap). Good enough to see discovery + filtering behavior.")
else:
    from inhouse_wrappers import InHouseEmbeddings
    embedder = InHouseEmbeddings()
    print("Real InHouseEmbeddings().")

## 1. Load the API catalog

In [ ]:
df = pd.read_csv("dataset/api_catalog.csv")
print(f"{len(df)} endpoints, columns: {list(df.columns)}")
META_FIELDS = ["endpoint", "method", "domain", "auth", "version", "status"]
df.head(8)

## 2. Ingest into Chroma with discovery metadata
`text` (the purpose) gets embedded; the rest becomes filterable metadata.

In [ ]:
client = chromadb.PersistentClient(path="./api_discovery_db")
try: client.delete_collection("api_catalog")
except Exception: pass
coll = client.get_or_create_collection("api_catalog", metadata={"hnsw:space": "cosine"})

ids = df["id"].tolist()
docs = df["text"].tolist()
metadatas = [{f: row[f] for f in META_FIELDS} for _, row in df.iterrows()]
embeddings = embedder.embed_documents(docs)

coll.add(ids=ids, embeddings=embeddings, documents=docs, metadatas=metadatas)
print(f"Ingested {coll.count()} endpoints.")
print("Example:", coll.get(ids=['pay_refund_01'], include=['documents','metadatas'])['metadatas'][0])

## 3. Discovery retrieval — the core use case
Ask in plain language, get the right endpoint back. This is what the whole system is for.

In [ ]:
def discover(query, k=3, where=None):
    qv = embedder.embed_query(query)
    kw = {"query_embeddings": [qv], "n_results": k, "include": ["documents", "metadatas", "distances"]}
    if where: kw["where"] = where
    res = coll.query(**kw)
    out = []
    for i in range(len(res["ids"][0])):
        out.append({
            "id": res["ids"][0][i],
            "endpoint": res["metadatas"][0][i]["endpoint"],
            "method": res["metadatas"][0][i]["method"],
            "domain": res["metadatas"][0][i]["domain"],
            "purpose": res["documents"][0][i],
            "distance": round(res["distances"][0][i], 3),
        })
    return pd.DataFrame(out)

print("Q: 'how do I refund a payment?'")
discover("how do I refund a payment")

**What good looks like:** `pay_refund_01` (`POST /v1/payments/{id}/refund`) should be the top hit. With the mock embedder the ranking is approximate; with real Jina it's sharp. Try a few of your own queries in the next cell.

In [ ]:
for q in ["freeze a lost card", "get an access token", "open a new bank account",
          "send an SMS to a customer", "move money between accounts"]:
    top = discover(q, k=1).iloc[0]
    print(f"{q:38s} -> {top['method']:6s} {top['endpoint']:32s} ({top['domain']})")

## 4. Filtered discovery — metadata narrows the search
This is the payoff of the `domain`/`method` metadata: constrain discovery to a domain or HTTP method. Exactly the filters Chroma Studio exposes in its UI.

In [ ]:
print("Q: 'create something' — but ONLY in the cards domain:")
display(discover("create something new", k=3, where={"domain": "cards"}))

print("\nQ: 'get details' — but ONLY GET endpoints:")
display(discover("get details of a resource", k=3, where={"method": "GET"}))

print("\nQ: 'payment' — POST endpoints in the payments domain (combined filter):")
display(discover("payment", k=3, where={"$and": [{"domain": "payments"}, {"method": "POST"}]}))

**Why this matters for API discovery specifically:** a query like 'create' matches many endpoints across domains. Filtering by `domain=cards` turns a vague semantic match into a precise, scoped answer — which is exactly how a developer narrows down in practice ('I know it's a card operation, show me card creates').

## 5. Examination — analyze the catalog like you would in Chroma Studio
Now we switch from *using* the collection to *examining* it — the analyze→decide→act loop. Each check below maps to a Chroma Studio view.

### 5a. Metadata summary (Studio: Browse → Metadata summary panel)

In [ ]:
def scan_metadata(coll):
    got = coll.get(limit=coll.count(), include=["metadatas"])
    metas = got["metadatas"]
    keys = sorted({k for m in metas if m for k in m.keys()})
    summary = {}
    for k in keys:
        c = {}
        for m in metas:
            if m and k in m: c[str(m[k])] = c.get(str(m[k]), 0) + 1
        summary[k] = dict(sorted(c.items(), key=lambda x: -x[1]))
    return summary

for k, counts in scan_metadata(coll).items():
    print(f"{k:9s}: {counts}")

**Decision surfaced:** `status` shows one `deprecated` endpoint, and `method` is heavily `POST`. In Studio you'd see this instantly in the summary panel — it tells you the catalog's shape before you retrieve anything.

### 5b. Coverage map (Studio: Visualize → color by domain)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

got = coll.get(limit=coll.count(), include=["documents","metadatas","embeddings"])
vecs = np.array(got["embeddings"])
metas = got["metadatas"]; ids = got["ids"]
coords = PCA(n_components=2).fit_transform(vecs)

fig, ax = plt.subplots(figsize=(10,7))
domains = sorted(set(m["domain"] for m in metas))
cmap = plt.get_cmap("tab10")
for di, dom in enumerate(domains):
    idx = [i for i,m in enumerate(metas) if m["domain"]==dom]
    ax.scatter(coords[idx,0], coords[idx,1], color=cmap(di), label=dom, s=80, alpha=0.8)
for i, _id in enumerate(ids):
    ax.annotate(_id, (coords[i,0], coords[i,1]), fontsize=6, alpha=0.6)
ax.legend(title="domain"); ax.set_title("API catalog embedding space (PCA, colored by domain)")
plt.tight_layout(); plt.show()

**What to look for (same as Studio's Visualize tab):** do endpoints of the same domain cluster together? Where a `notifications` point sits among `customers`, that's the cross-domain overlap we planted (`notif_prefs_01` lives at a customer path). That's a *design* insight the map hands you.

### 5c. Near-duplicate detection (Studio: Visualize → stacked points)

In [ ]:
sim = vecs @ vecs.T
np.fill_diagonal(sim, 0)
print("Endpoint pairs with very similar PURPOSE (cosine > 0.5):")
for i in range(len(ids)):
    for j in range(i+1, len(ids)):
        if sim[i,j] > 0.5:
            print(f"  {ids[i]:18s} <-> {ids[j]:18s}  sim={sim[i,j]:.2f}")

**Decision surfaced:** `pay_create_01` vs `pay_create_02` (v1 vs v2 of create) show up as near-duplicates. That's expected — but it's a *decision point*: do you want both versions discoverable, or should discovery return only the latest (v2) and hide v1? In Studio you'd spot the stacked points and decide. A common answer: keep both but filter `version` at query time so callers get the version they target.

### 5d. The 'wrong result' failure drill (Studio: Search → read the ranked list)

In [ ]:
print("Q: 'notify my application of events' (should find the WEBHOOK, not send-notification)")
res = discover("notify my application of events", k=3)
display(res)

**The expert habit:** `notif_send_01` (send an SMS/email) and `notif_webhook_01` (register a webhook) both mention 'notification' and read alike — the planted collision. If the wrong one ranks first, the fix isn't to blame the model: it's to add a discriminating metadata field (e.g. `pattern: push` vs `pattern: webhook`) or sharpen the purpose text. Reading the *ranked list with distances* is how you catch this — exactly what Studio's Search tab shows.

### 5e. Deprecated-endpoint hygiene (Studio: Browse filter status=deprecated)

In [ ]:
dep = coll.get(where={"status": "deprecated"}, include=["documents","metadatas"])
print("Deprecated endpoints that should probably be EXCLUDED from discovery:")
for i, doc in zip(dep["ids"], dep["documents"]):
    print(f"  {i}: {doc}")
print("\nDecision: filter status='active' at query time so discovery never surfaces")
print("the legacy /v1/login. In Studio you'd confirm the count via the Browse filter.")

# demonstrate the fix:
print("\nSame query WITHOUT vs WITH the active-only filter:")
print("  unfiltered top hit:", discover("log in with username and password", k=1).iloc[0]["id"])
print("  active-only  top hit:", discover("log in with username and password", k=1,
      where={"status": "active"}).iloc[0]["id"])

## 6. What you just built (and how it maps to Chroma Studio)
| Notebook step | Chroma Studio equivalent | The decision it drives |
|---|---|---|
| Ingest with metadata | Add tab | — |
| Discovery retrieval | Search tab | Does semantic discovery return the right API? |
| Filtered discovery | Browse/Visualize metadata filter | Scope discovery to a domain/method |
| Metadata summary | Browse summary panel | Understand catalog shape |
| Coverage map | Visualize, color by domain | Do domains cluster? cross-domain overlap? |
| Near-duplicates | Visualize stacked points | v1/v2 — keep both or hide old? |
| Failure drill | Search ranked list | Fix collisions with metadata, not blame |
| Deprecated hygiene | Browse filter status | Exclude deprecated from discovery |

**Open this collection in Chroma Studio** (point its folder at the `api_discovery_db` path printed below) and reproduce sections 5a–5e visually. The notebook is the 'why'; Studio is the interactive 'where you'd click'.

In [ ]:
import os
print("Chroma Studio -> sidebar -> Local Chroma folder:")
print("  ", os.path.abspath("./api_discovery_db"))
print("Then select the 'api_catalog' collection.")
print("\nTry in Studio:")
print("  Browse: filter domain=payments  -> should show 6 endpoints")
print("  Visualize: color by domain, filter method=GET -> the read endpoints only")
print("  Search: 'refund a payment' -> pay_refund_01 on top")

## Phase 2 preview (separate, later)
Phase 1 discovers *which* endpoint. Phase 2 will add, per endpoint: request parameters, body schema, response schema, and error codes — chunked so a developer can ask *'what fields does the refund endpoint take?'* and get the schema. That's a different chunking design (one endpoint → several schema chunks) and will be its own dataset + notebook, building on this one.